## Bound Entanglement
### 1. 随机初始化 Ansatz
### 2. 构造Bound Entangled State

In [ ]:
import torch
import numpy as np
import sys
import matplotlib.pyplot as plt
import seaborn as sns

# ================= 配置 =================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64) 
np.set_printoptions(threshold=sys.maxsize, linewidth=200, precision=6, suppress=True)
print(f"Running on: {device}")


# ================= 函数 =================

def get_ansatz(dim, rank):           #构造参数矩阵A，维度为 (dim^2, rank)，元素为复数
    real = torch.randn(dim * dim, rank, device=device)
    imag = torch.randn(dim * dim, rank, device=device)
    A = torch.complex(real, imag)
    A.requires_grad = True
    return A


def get_rho(A):                # Cholesky decomposition构造密度矩阵
    rho_un = A @ A.mH
    trace = torch.trace(rho_un) + 1e-16 # 防止trace为0
    return rho_un / trace


def partial_transpose_torch(rho, dim):          # 计算部分转置，判断量子态是否为PPT态
    rho_reshaped = rho.view(dim, dim, dim, dim)
    rho_pt = rho_reshaped.permute(0, 3, 2, 1).contiguous()   # 部分转置操作
    return rho_pt.view(dim * dim, dim * dim)



def get_metrics(rho, dim):

    # =====PPT最小特征值======
    pt = partial_transpose_torch(rho, dim)
    min_ppt = torch.min(torch.linalg.eigvalsh(pt))

    # ======调整CCN范数=======
    rho_reshaped = rho.view(dim, dim, dim, dim)
    rho_r = rho_reshaped.permute(0, 2, 1, 3).contiguous().view(dim*dim, dim*dim)  # 矩阵重排操作
    r_norm = torch.linalg.matrix_norm(rho_r, ord='nuc')  # trace norm (核范数) 作为纠缠度量

    return min_ppt, r_norm



# ================= 注入白噪声 =================

def apply_white_noise(rho_np, dim, p):
    """
    rho_new = (1 - p) * rho + p * (I / D
    rho : 原始密度矩阵 (Numpy array)
    D : 单个子系统的维度 (总维度为 dim * dim)
    p : 白噪声比例 [0, 1]
    """

    if not (0.0 <= p <= 1.0):
        raise ValueError("白噪声比例 p 必须在 0 到 1 之间")

    D = dim * dim
    
    I_normalized = np.eye(D) / D  
    
    rho_repaired = (1 - p) * rho_np + p * I_normalized
    
    return rho_repaired

# ================= 生成束缚纠缠态 =================

TASK_DIM = 4           # 量子比特维度 (2-10)
TASK_RANK = 4        # Ansatz秩
TASK_MAX_ATTEMPTS = 30 # 最大尝试次数

def generate_be_robust(dim=TASK_DIM, rank=TASK_RANK, max_attempts=TASK_MAX_ATTEMPTS):
    print(f"\n 目标: 构造 Dim {dim}x{dim}, Rank {rank} ")
    
    for attempt in range(1, max_attempts + 1):
        A = get_ansatz(dim, rank)
        optimizer = torch.optim.Adam([A], lr=0.04)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=500)
        
        print(f"\n[Attempt {attempt}]")
        
        # 稍微放宽判定标准，捕捉“差一点点”的态
        best_candidate = None
        best_candidate_score = -1.0
        
        total_steps = 6000
        start_target = 1.01
        end_target = 1.0005 # 稍微高一点，留给修复用
        
        for i in range(total_steps):
            optimizer.zero_grad()
            rho = get_rho(A)
            min_pt, r_norm = get_metrics(rho, dim)
            
    
            # 如果 PPT 稍微负一点点 (>-0.001) 且 纠缠度还不错 (>1.002)
            # 存下来，最后尝试修复
            if min_pt.item() > -0.001 and r_norm.item() > 1.002:
                # 评分标准：PPT越接近0越好，Norm越大越好
                score = r_norm.item() - 100 * abs(min_pt.item())
                if score > best_candidate_score:
                    best_candidate_score = score
                    best_candidate = rho.detach().cpu().numpy()

            # --- 损失函数 ---
            decay = np.exp(-5.0 * i / total_steps)
            current_target = end_target + (start_target - end_target) * decay
            
            loss = 0
            ppt_weight = 1000.0 + (i / total_steps) * 9000.0
            
            if min_pt < 0:
                loss += torch.nn.functional.softplus(-min_pt * ppt_weight)
            else:
                loss += -min_pt * 0.1

            if r_norm < current_target:
                diff = current_target - r_norm
                ent_weight = 100.0 + (i / total_steps) * 400.0
                loss += diff * ent_weight
            
            loss += 0.0001 * torch.norm(A)
            
            loss.backward()
            optimizer.step()
            scheduler.step(loss)
            
            # 完美成功判定
            if min_pt.item() > -1e-8 and r_norm.item() > 1.00001:
                print(f"\n [PERFECT HIT] Step {i}")
                return rho.detach().cpu().numpy()
            
            if i % 1000 == 0:
                print(f"   Step {i:4d} | Target: {current_target:.5f} | PPT: {min_pt.item():.6f} | Norm: {r_norm.item():.6f}")
        
        # 循环结束，如果没有完美结果，尝试修复最佳候选者
        if best_candidate is not None:
            print("   >>> 尝试修复最佳候选者...")
            rho_fixed = try_repair_state(best_candidate, dim)
            
            # 验证修复结果
            rho_t = torch.tensor(rho_fixed, device=device)
            m_ppt, m_norm = get_metrics(rho_t, dim)
            
            print(f"   >>> 修复后: PPT={m_ppt.item():.8f}, Norm={m_norm.item():.8f}")
            
            if m_ppt.item() > -1e-15 and m_norm.item() > 1.0:
                print("🎉 [REPAIR SUCCESS] 成功修复为束缚纠缠态！")
                return rho_fixed
            else:
                print("   >>> 修复后纠缠度丢失，继续尝试...")
        else:
            print("   >>> 没有找到有修复价值的候选者。")

    print("\n All attempts failed.")
    return None

# ================= 运行 =================

rho_final = generate_be_robust(dim=TASK_DIM, rank=TASK_RANK, max_attempts=TASK_MAX_ATTEMPTS) 

if rho_final is not None:
    print("\n" + "="*60)
    print(" 最终结果 ")
    print("="*60)
    
    rho_np = rho_final
    
    # 打印矩阵供复制
    print("rho = np.array(")
    print(np.array2string(rho_np, separator=', '))
    print(")")
    
    # 最终严格物理验证
    dim = TASK_DIM
    rank = TASK_RANK
    
    # PPT 验证
    rho_ts = rho_np.reshape(dim, dim, dim, dim)
    rho_pt = rho_ts.transpose(0, 3, 2, 1).reshape(dim*dim, dim*dim)
    min_eig = np.min(np.linalg.eigvalsh(rho_pt))
    
    # 纠缠验证
    rho_r = rho_ts.transpose(0, 2, 1, 3).reshape(dim*dim, dim*dim)
    r_norm = np.sum(np.linalg.svd(rho_r, compute_uv=False))
    
    # Rank 验证
    s = np.linalg.svd(rho_np, compute_uv=False)
    # 因为修复过程加了满秩的白噪声，Rank 会变成 Full Rank (16)
    # 但白噪声极其微小 (1e-6级别)，所以主要成分的 Rank 还是 4
    # 我们看 "显著奇异值"
    eff_rank = np.sum(s > 1e-4) 
    
    print("-" * 30)
    print(f"1. PPT Check (Min Eig) : {min_eig:.9f}  [{'PASS' if min_eig > -1e-9 else 'FAIL'}]")
    print(f"2. Entanglement (Norm) : {r_norm:.9f}  [{'PASS' if r_norm > 1.0 else 'FAIL'}]")
    print(f"3. Significant Rank    : {eff_rank}          (Base Rank was {rank})")
    print("-" * 30)
    
    if min_eig > -1e-9 and r_norm > 1.0:
        print("结论: 这是一个有效的束缚纠缠态。")
    else:
        print("结论: 数据仍有瑕疵。")

# 确保 rho_final 是 numpy 格式
if isinstance(rho_final, torch.Tensor):
    rho_np = rho_final.detach().cpu().numpy()
else:
    rho_np = rho_final

# 文本打印
print("\n" + "="*50)
print("密度矩阵数值展示")
print("="*50)

# 设置打印选项
np.set_printoptions(threshold=np.inf, linewidth=200, precision=4, suppress=True)

# 打印实部
print("--- 实部 (Real Part) ---")
print(rho_np.real)

# 打印虚部
if np.max(np.abs(rho_np.imag)) > 1e-5:
    print("\n--- 虚部 (Imaginary Part) ---")
    print(rho_np.imag)
else:
    print("\n(虚部几乎为0，忽略不计)")


# 可视化 (Cityscape/Heatmap)
plt.figure(figsize=(12, 5))

# 实部热力图
plt.subplot(1, 2, 1)
sns.heatmap(rho_np.real, cmap="RdBu_r", center=0, square=True)
plt.title(f"Real Part (Dim=4, Rank=4 BE State)\nNorm={1.0049:.5f}")

# 虚部热力图
plt.subplot(1, 2, 2)
sns.heatmap(rho_np.imag, cmap="RdBu_r", center=0, square=True)
plt.title("Imaginary Part")

plt.tight_layout()
plt.show()

# 保存到文件

filename = f"BE_State_{TASK_DIM}x{TASK_DIM}_Rank{TASK_RANK}_7.npy"
np.save(filename, rho_np)
print(f"\n 矩阵已保存至: {filename}")
print(f"读取方法: rho = np.load('{filename}')")

#  验证是否为 PPT (再次确认)

def check_ppt(rho):
    dim = int(np.sqrt(rho.shape[0]))
    rho_tensor = rho.reshape(dim, dim, dim, dim)
    # 对第二个子系统 (B) 进行转置 (索引 1, 3)
    rho_pt = rho_tensor.transpose(0, 3, 2, 1).reshape(dim*dim, dim*dim)
    min_eig = np.min(np.linalg.eigvalsh(rho_pt))
    return min_eig

ppt_val = check_ppt(rho_np)
print(f"\n 最终 PPT 检查: Min Eig = {ppt_val:.9f}")
if ppt_val > -1e-6:
    print("✅ 这是一个合格的 PPT 态。")
else:
    print("⚠️ 警告：仍有微小负特征值，可能需要更多白噪声混合。")



KeyboardInterrupt: 

In [ ]:
import numpy as np

# 假设你的矩阵已经命名为 rho (即你上面提供的代码)

# 1. 验证偏转置 (Partial Transpose, PPT判据)
dimA, dimB = 4, 4
rho_tensor = rho.reshape(dim, dim, dim, dim)

# 对子系统 B 求偏转置: 交换第二个和第四个索引 (即1和3)
rho_pt_tensor = rho_tensor.transpose((0, 3, 2, 1))
rho_pt = rho_pt_tensor.reshape((16, 16))

# 计算偏转置矩阵的特征值
eigenvalues_pt = np.linalg.eigvalsh(rho_pt)
min_eig = np.min(eigenvalues_pt)

print(f"偏转置后的最小特征值: {min_eig:.8f}")

TOLERANCE = -1e-8 # 容忍机器计算的数值误差
if min_eig < TOLERANCE:
    print("\n👉 【结论】：这不是束缚纠缠态！")
    print("该态具有负偏转置（NPT）。它是可提纯的【自由纠缠态】（Free Entangled）。")
else:
    print("该态具有正偏转置（PPT）。正在使用 CCNR 判据检测是否纠缠...")
    
    # 2. CCNR判据 (交叉范数重排判据)，用于检测 PPT 态是否真正纠缠
    R_tensor = rho_tensor.transpose((0, 2, 1, 3))
    R_matrix = R_tensor.reshape((16, 16))
    singular_values = np.linalg.svd(R_matrix, compute_uv=False)
    ccnr_value = np.sum(singular_values)
    
    print(f"CCNR 判据值 (Trace Norm): {ccnr_value:.6f}")
    
    if ccnr_value > 1.0 + TOLERANCE:
        print("\n👉 【结论】：确认！这是一个确凿的【两体束缚纠缠态】(Bound Entangled State)！")
        print("它不仅具有正偏转置(PPT)，同时被证明是不可分(纠缠)的。")
    else:
        print("\n👉 【结论】：它是PPT态，但CCNR判据值 <= 1。")
        print("这表示它内部的噪声可能过大，大概率已经退化成了【可分态】（Separable State）。如果要证明它是束缚纠缠，需要你使用当时跑代码时的专属纠缠目击者（Entanglement Witness）来验证。")

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

# ==========================================
# 1. 生成 Horodecki 态 (d=3)
# ==========================================

def generate_horodecki_state_np(a=0.3):
    """
    根据 arXiv:1101.5754 生成 3x3 的 Horodecki PPT 纠缠态。
    这是纯 Numpy 实现，方便后续检验。
    """
    if not (0 <= a <= 1):
        raise ValueError("参数 a 必须在 [0, 1] 之间")
    
    b = (1 + a) / 2
    c = np.sqrt(1 - a**2) / 2
    
    rho = np.zeros((9, 9), dtype=complex)
    
    # 填充对角线
    rho[0,0]=a; rho[1,1]=a; rho[2,2]=a; rho[3,3]=a; rho[4,4]=a; rho[5,5]=a
    rho[6,6]=b; rho[7,7]=a; rho[8,8]=b
    
    # 填充非对角线
    rho[0,4]=a; rho[0,8]=a; rho[4,0]=a; rho[4,8]=a; rho[8,0]=a; rho[8,4]=a
    rho[6,8]=c; rho[8,6]=c
    
    # 归一化
    rho = rho / (8*a + 1)
    return rho

# [重要设定]：生成态并赋值给全局变量，供检验代码读取
target_dim = 3
# 我们选择 a=0.5，这是 Horodecki 态纠缠最强的参数点
rho_np = generate_horodecki_state_np(a=0.5) 

print("✅ Horodecki 态 (a=0.5, d=3) 已成功生成并加载至内存！\n")

# ==========================================
# 2. 你的检验程序 (完整保留你的逻辑)
# ==========================================

def check_bound_entanglement_in_notebook():
    # ================= 配置区 =================
    try:
        if 'rho_final' in globals() and rho_final is not None:
            rho_input = rho_final
        elif 'rho_np' in globals() and rho_np is not None:
            rho_input = rho_np
        else:
            print("❌ 错误：未找到生成的密度矩阵变量 (rho_final 或 rho_np)。")
            return

        if 'target_dim' in globals():
            d = target_dim
        else:
            d = int(np.sqrt(rho_input.shape[0]))
            print(f"⚠️ 未找到 target_dim 变量，自动推断维度为 d={d}")
            
    except Exception as e:
        print(f"❌ 读取数据时出错: {e}")
        return
    # ==========================================

    print(f"🔬 正在分析 {d}x{d} 维量子态 (Matrix: {d**2}x{d**2})...\n")

    # 1. 格式标准化 (Tensor -> Numpy, 厄米化, 归一化)
    if isinstance(rho_input, torch.Tensor):
        rho = rho_input.detach().cpu().numpy()
    else:
        rho = np.array(rho_input, copy=True)
    
    # 强行厄米化 (消除计算误差)
    rho = (rho + rho.conj().T) / 2
    # 强行归一化
    rho = rho / np.trace(rho)

    d2 = d * d
    tolerance = 1e-9 # 浮点数容差

    # ---------------------------------------------------
    # 2. 判据 A: PPT (Positive Partial Transpose)
    # ---------------------------------------------------
    rho_tensor = rho.reshape(d, d, d, d)
    # 转置子系统B (交换下标 1, 3)
    rho_pt = rho_tensor.transpose(0, 3, 2, 1).reshape(d2, d2)
    pt_evals = np.linalg.eigvalsh(rho_pt)
    min_pt_eig = np.min(pt_evals)

    is_ppt = min_pt_eig >= -tolerance

    # ---------------------------------------------------
    # 3. 判据 B: CCN (Computable Cross Norm / Realignment)
    # ---------------------------------------------------
    # 重排: indices (i, j, k, l) -> (i, k, j, l)
    rho_r = rho_tensor.transpose(0, 2, 1, 3).reshape(d2, d2)
    # 计算奇异值之和 (Trace Norm)
    singular_values = np.linalg.svd(rho_r, compute_uv=False)
    ccn_norm = np.sum(singular_values)

    is_entangled = ccn_norm > 1.0 + tolerance

    # ---------------------------------------------------
    # 4. 输出结果报告
    # ---------------------------------------------------
    print(f"1️⃣  [PPT 判据] 部分转置最小特征值: {min_pt_eig:.9f}")
    if is_ppt:
        print("    -> 结果: ✅ PPT (正部分转置)")
        print("    -> 含义: 该态不可被提纯。")
    else:
        print("    -> 结果: ❌ NPT (负部分转置)")
        print("    -> 含义: 该态是自由纠缠态 (Free Entangled)，不是束缚纠缠。")

    print(f"\n2️⃣  [CCN 判据] 重排范数 (Target > 1): {ccn_norm:.9f}")
    if is_entangled:
        print("    -> 结果: ✅ 确认纠缠")
    else:
        print("    -> 结果: ❓ 未检测到纠缠 (可能为可分态)")

    print("\n" + "="*40)
    if is_ppt and is_entangled:
        print("🏆 最终结论: 这是一个完美的【束缚纠缠态】! 🏆")
    elif not is_ppt:
        print("结论: 这是一个【自由纠缠态】 (NPT Entangled)。")
    else:
        print("结论: 无法确认纠缠 (可能是PPT可分态)。")
    print("="*40)

    # ---------------------------------------------------
    # 5. 可视化绘图
    # ---------------------------------------------------
    plt.figure(figsize=(12, 4))
    
    # 图1: 部分转置谱 (PT Eigenvalues)
    plt.subplot(1, 2, 1)
    # 给特征值排序后画图，看着更直观
    sorted_evals = np.sort(pt_evals)
    plt.plot(sorted_evals, 'b.-', markersize=8, label='PT Eigenvalues')
    plt.axhline(0, color='r', linestyle='--', linewidth=1.5, label='Zero Boundary')
    plt.title(f"PT Spectrum (Min Eig: {min_pt_eig:.1e})", fontsize=12)
    plt.xlabel("Eigenvalue Index")
    plt.ylabel("Value")
    plt.legend()
    plt.grid(True, alpha=0.4)
    
    # 图2: 密度矩阵实部热力图
    plt.subplot(1, 2, 2)
    # 对于 Horodecki 态，取一个合适的显示范围
    sns.heatmap(rho.real, cmap="RdBu_r", center=0, vmin=-0.15, vmax=0.15, 
                annot=True, fmt=".2f", annot_kws={"size": 7}) # 添加数字标注方便查看
    plt.title("Density Matrix Real Part (Horodecki a=0.5)", fontsize=12)
    
    plt.tight_layout()
    plt.show()

# ==========================================
# 3. 运行！
# ==========================================
check_bound_entanglement_in_notebook()

## 产生 Bound Entangled State 混合初始化 Hybrid Ansatz

In [ ]:
import torch
import numpy as np
import sys
import time
import random
import matplotlib.pyplot as plt
import seaborn as sns

# ================= 配置 =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64) 
np.set_printoptions(threshold=sys.maxsize, linewidth=200, precision=5, suppress=True)

print(f"Running on: {device}")

TASK_DIM = 7           # 量子比特维度 (2-10)
TASK_RANK = 4          # Ansatz秩
TASK_MAX_ATTEMPTS = 20 # 最大尝试次数

# ================= 基础函数 =================

def get_ansatz_hybrid(dim, rank):                 
    """
    混合初始化：
    不仅仅是随机，而是让初始态稍微接近 Mixed State (I/d)，
    这样离 PPT 区域更近，减少优化难度。
    """
    # 随机部分
    real = torch.randn(dim * dim, rank, device=device)
    imag = torch.randn(dim * dim, rank, device=device)
    A_rand = torch.complex(real, imag) # 随机复数矩阵
    
    # 单位矩阵部分 (Reshape 成列向量形式混合)
    # 这相当于给 ansatz 加一个 bias，使其不过于极端
    bias = torch.eye(dim * dim, rank, device=device) * 0.5
    
    A = A_rand + bias
    A.requires_grad = True
    return A

def get_rho(A):
    rho_un = A @ A.mH
    trace = torch.trace(rho_un) + 1e-16
    return rho_un / trace

def partial_transpose(rho, dim):
    rho_reshaped = rho.view(dim, dim, dim, dim)
    rho_pt = rho_reshaped.permute(0, 3, 2, 1).contiguous()
    return rho_pt.view(dim * dim, dim * dim)

def get_metrics_high_dim(rho, dim):
    """
    [改进2] 针对高维的指标计算
    返回: 
    1. min_eig: 最小特征值 (用于最终判断)
    2. negativity_sum: 所有负特征值之和 (用于提供更强的梯度)
    3. r_norm: 重排范数
    """
    # PPT 相关
    pt = partial_transpose(rho, dim)
    eigvals = torch.linalg.eigvalsh(pt)
    
    min_eig = torch.min(eigvals)
    # 计算所有负特征值的平方和 (作为 Loss 梯度更平滑)
    # 或者直接用 sum(abs(negative_eigs))
    negatives = eigvals[eigvals < 0]
    negativity_loss = torch.sum(torch.square(negatives)) 
    
    # Entanglement 相关
    rho_reshaped = rho.view(dim, dim, dim, dim)
    rho_r = rho_reshaped.permute(0, 2, 1, 3).contiguous().view(dim*dim, dim*dim)
    r_norm = torch.linalg.matrix_norm(rho_r, ord='nuc')
    
    return min_eig, negativity_loss, r_norm

# ================= 修复函数 =================
def try_repair_state(rho_np, dim):
    print("   >>> [Auto-Repair] 启动微量白噪声混合...")
    d2 = dim * dim
    I = np.eye(d2) / d2
    
    rho_tensor = rho_np.reshape(dim, dim, dim, dim)
    rho_pt = rho_tensor.transpose(0, 3, 2, 1).reshape(d2, d2)
    min_eig = np.min(np.linalg.eigvalsh(rho_pt))
    
    if min_eig >= 0: return rho_np
    
    # 动态计算需要的噪声比例
    p_needed = abs(min_eig) * d2 * 1.2 # 给 1.2 倍余量
    print(f"   >>> 需要混合 p = {p_needed:.8f}")
    
    rho_repaired = (1 - p_needed) * rho_np + p_needed * I
    return rho_repaired

# ================= 核心：高维优化器 =================
def set_seed(seed):
    """固定所有随机种子，确保可复现"""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # 牺牲一点速度换取确定性（可选）
        # torch.backends.cudnn.deterministic = True
        # torch.backends.cudnn.benchmark = False

def solve_high_dim_optimized(dim=TASK_DIM, rank=TASK_RANK, max_attempts=TASK_MAX_ATTEMPTS, base_seed=42):
    print(f"\n🚀 [优化版] 启动高维攻坚: Dim {dim}x{dim}, Rank {rank}")
    print(f"🔧 策略: 动态两阶段优化 + 随机种子锁定 (Base Seed: {base_seed})")

    for attempt in range(1, max_attempts + 1):
        # 1. 设定当前尝试的种子
        current_seed = base_seed + attempt
        set_seed(current_seed)
        
        # 2. 初始化
        A = get_ansatz_hybrid(dim, rank)
        
        # 使用 AdamW
        optimizer = torch.optim.AdamW([A], lr=0.005, weight_decay=1e-5)
        
        # 增加步数，因为我们要进行精细微调
        total_steps = 20000 
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-5)
        
        print(f"\n[Attempt {attempt}/{max_attempts}] Seed: {current_seed}")
        
        best_candidate = None
        best_r_norm = 0
        best_min_eig = -1.0
        
        start_time = time.time()
        
        for i in range(total_steps):
            optimizer.zero_grad()
            rho = get_rho(A)
            
            # 计算指标
            # min_eig: PPT判据 (目标 >= 0)
            # neg_loss: 负特征值平方和 (优化动力)
            # r_norm: CCN判据 (目标 > 1)
            min_eig, neg_loss, r_norm = get_metrics_high_dim(rho, dim)
            
            # --- 记录最佳结果 ---
            # 条件：PPT 几乎满足 且 纠缠度高
            if min_eig.item() > -1e-5 and r_norm.item() > 1.0:
                # 只有当它是 PPT 且比之前的纠缠度更高时才保存
                if r_norm.item() > best_r_norm:
                    best_r_norm = r_norm.item()
                    best_min_eig = min_eig.item()
                    best_candidate = rho.detach().cpu().numpy()
            
            # --- 动态 Loss 设计 (核心优化) ---
            
            # 阶段判定：如果 PPT 已经合格 (>-1e-6)，则进入“贪婪纠缠模式”
            is_ppt_satisfied = min_eig.item() > -1e-6
            
            loss = 0.0
            
            if not is_ppt_satisfied:
                # [阶段一]：生存模式。首要任务是满足 PPT。
                # 强力惩罚负特征值，同时也给一点点奖励给 r_norm 防止坍缩成单位阵
                ppt_weight = 10000.0 * (1 + i/5000) # 随时间增加惩罚
                loss += neg_loss * ppt_weight
                loss += -r_norm * 0.1 # 弱引导
            else:
                # [阶段二]：贪婪模式。PPT 已满足，全力寻找更强的纠缠。
                # 只要 min_eig 还在安全区，就疯狂最大化 r_norm
                
                # 这里引入“安全边界”惩罚：虽然 > 0，但如果太靠近 0 也不好，稍微推远一点
                # 但主要权重给 r_norm
                loss += -r_norm * 50.0  # 强力推高 CCN
                
                # 保持 PPT 的约束 (Barrier Method)
                # 如果掉回负数，这个项会瞬间变大
                if min_eig < 0:
                    loss += neg_loss * 50000.0 
            
            # 反向传播
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            # --- 提前终止与日志 ---
            if i % 5000 == 0:
                print(f"   Step {i:5d} | PPT_Min: {min_eig.item():.7f} | Norm: {r_norm.item():.7f} | Mode: {'🟢Greedy' if is_ppt_satisfied else '🔴Survival'}")
            
            # 成功判定：非常严格的标准
            if min_eig.item() > -1e-9 and r_norm.item() > 1.0001:
                duration = time.time() - start_time
                print(f"\n🎉 [FOUND!] Step {i} (Seed {current_seed})")
                print(f"   耗时: {duration:.2f}s")
                print(f"   指标: PPT={min_eig.item():.9f}, CCN={r_norm.item():.9f}")
                return rho.detach().cpu().numpy()
        
        # 本次尝试结束，检查是否有幸存的“次优解”并尝试修复
        if best_candidate is not None:
            print(f"   ⚠️ 尝试修复最佳候选 (PPT={best_min_eig:.6f}, Norm={best_r_norm:.6f})...")
            rho_fixed = try_repair_state(best_candidate, dim)
            
            # 验证修复后的状态
            # 注意：修复通常会降低 Norm，所以必须再次检查
            rho_t = torch.tensor(rho_fixed, device=device)
            m_eig, _, m_norm = get_metrics_high_dim(rho_t, dim)
            
            if m_eig.item() > -1e-15 and m_norm.item() > 1.0:
                print(f"   ✨ 修复成功! Norm={m_norm.item():.6f}")
                return rho_fixed
            else:
                print(f"   💨 修复失败 (Norm降至 {m_norm.item():.6f})")

    print("\n💀 所有尝试均未找到完美符合条件的态。")
    return None

# ================= 运行 =================

target_dim =TASK_DIM
target_rank = TASK_RANK

rho_final = solve_high_dim_optimized(target_dim, target_rank, max_attempts=TASK_MAX_ATTEMPTS)

if rho_final is not None:
    print("\n" + "="*50)
    print("最终验证报告")
    print("="*50)
    
    rho_np = rho_final
    dim = target_dim
    
    # 1. PPT Check
    rho_ts = rho_np.reshape(dim, dim, dim, dim)
    rho_pt = rho_ts.transpose(0, 3, 2, 1).reshape(dim*dim, dim*dim)
    min_eig = np.min(np.linalg.eigvalsh(rho_pt))
    
    # 2. Entanglement Check
    rho_r = rho_ts.transpose(0, 2, 1, 3).reshape(dim*dim, dim*dim)
    r_norm = np.sum(np.linalg.svd(rho_r, compute_uv=False))
    
    # 3. Rank Check
    s = np.linalg.svd(rho_np, compute_uv=False)
    eff_rank = np.sum(s > 1e-4) # 稍微放宽阈值以过滤白噪声
    
    print(f"Dimension: {dim}x{dim}")
    print(f"PPT Min Eig: {min_eig:.9f}  [{'PASS' if min_eig > -1e-9 else 'FAIL'}]")
    print(f"Realignment: {r_norm:.9f}  [{'PASS' if r_norm > 1.0 else 'FAIL'}]")
    print(f"Approx Rank: {eff_rank}          (Target {target_rank})")
    
    if min_eig > -1e-9 and r_norm > 1.0:
        print("\n🏆 成功构造高维束缚纠缠态！")
        # 保存
        np.save(f"BE_State_d{dim}_r{target_rank}.npy", rho_np)
        print("矩阵已保存。")
    else:
        print("\n⚠️ 验证未完全通过。")

# ==========================================
# 1. 文本打印 (格式化输出)
# ==========================================
print("\n" + "="*50)
print("密度矩阵数值展示")
print("="*50)

# 设置打印选项：不省略，保留4位小数，一行显示更多
np.set_printoptions(threshold=np.inf, linewidth=200, precision=4, suppress=True)

# 打印实部
print("--- 实部 (Real Part) ---")
print(rho_np.real)

# 打印虚部 (如果有显著值)
if np.max(np.abs(rho_np.imag)) > 1e-5:
    print("\n--- 虚部 (Imaginary Part) ---")
    print(rho_np.imag)
else:
    print("\n(虚部几乎为0，忽略不计)")

# ==========================================
# 2. 可视化 (Cityscape/Heatmap)
# ==========================================
plt.figure(figsize=(12, 5))

# 实部热力图
plt.subplot(1, 2, 1)
sns.heatmap(rho_np.real, cmap="RdBu_r", center=0, square=True)
plt.title(f"Real Part (BE State)\nNorm={1.0049:.5f}")

# 虚部热力图
plt.subplot(1, 2, 2)
sns.heatmap(rho_np.imag, cmap="RdBu_r", center=0, square=True)
plt.title("Imaginary Part")

plt.tight_layout()
plt.show()

# ==========================================
# 3. 保存到文件
# ==========================================
filename = f"BE_State_{TASK_DIM}x{TASK_DIM}_Rank{TASK_RANK}.npy"
np.save(filename, rho_np)
print(f"\n 矩阵已保存至: {filename}")
print(f"读取方法: rho = np.load('{filename}')")

# ==========================================
# 4. 验证是否为 PPT (再次确认)
# ==========================================
def check_ppt(rho):
    dim = int(np.sqrt(rho.shape[0]))
    rho_tensor = rho.reshape(dim, dim, dim, dim)
    # 对第二个子系统 (B) 进行转置 (索引 1, 3)
    rho_pt = rho_tensor.transpose(0, 3, 2, 1).reshape(dim*dim, dim*dim)
    min_eig = np.min(np.linalg.eigvalsh(rho_pt))
    return min_eig

ppt_val = check_ppt(rho_np)
print(f"\n 最终 PPT 检查: Min Eig = {ppt_val:.9f}")
if ppt_val > -1e-6:
    print("✅ 这是一个合格的 PPT 态。")
else:
    print("⚠️ 警告：仍有微小负特征值，可能需要更多白噪声混合。")

Running on: cpu

🚀 [优化版] 启动高维攻坚: Dim 7x7, Rank 4
🔧 策略: 动态两阶段优化 + 随机种子锁定 (Base Seed: 42)

[Attempt 1/20] Seed: 43
   Step     0 | PPT_Min: -0.1168407 | Norm: 2.8760942 | Mode: 🔴Survival
   Step  5000 | PPT_Min: -0.0000135 | Norm: 0.9907686 | Mode: 🔴Survival
   Step 10000 | PPT_Min: -0.0000139 | Norm: 0.9959636 | Mode: 🔴Survival
   Step 15000 | PPT_Min: -0.0000082 | Norm: 0.9986466 | Mode: 🔴Survival

[Attempt 2/20] Seed: 44
   Step     0 | PPT_Min: -0.1162762 | Norm: 2.8765932 | Mode: 🔴Survival
   Step  5000 | PPT_Min: -0.0000115 | Norm: 0.9908310 | Mode: 🔴Survival
   Step 10000 | PPT_Min: -0.0000131 | Norm: 0.9968509 | Mode: 🔴Survival
   Step 15000 | PPT_Min: -0.0000144 | Norm: 0.9993428 | Mode: 🔴Survival

[Attempt 3/20] Seed: 45
   Step     0 | PPT_Min: -0.1247154 | Norm: 2.9402328 | Mode: 🔴Survival
   Step  5000 | PPT_Min: -0.0000121 | Norm: 0.9962984 | Mode: 🔴Survival
   Step 10000 | PPT_Min: -0.0000104 | Norm: 0.9980015 | Mode: 🔴Survival
   Step 15000 | PPT_Min: -0.0000070 | Norm: 0

KeyboardInterrupt: 

## 批量生成

In [ ]:
import torch
import numpy as np
import sys
import time
import random
import matplotlib.pyplot as plt
import seaborn as sns
import os  # 新增：用于文件路径管理

# ================= 配置与环境 =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64) 
np.set_printoptions(threshold=sys.maxsize, linewidth=200, precision=5, suppress=True)

print(f"Running on: {device}")

# ==========================================
#              1. 批量任务配置
# ==========================================
# 在这里输入你想跑的所有维度
TARGET_DIMS = [4]  
# 在这里输入你想跑的所有Rank
TARGET_RANKS = [4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16] 

# 输出文件夹名称
OUTPUT_DIR = "BE_States_Results"
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# ================= 基础函数 (保持不变) =================

def get_ansatz_hybrid(dim, rank):
    real = torch.randn(dim * dim, rank, device=device)
    imag = torch.randn(dim * dim, rank, device=device)
    A_rand = torch.complex(real, imag)
    bias = torch.eye(dim * dim, rank, device=device) * 0.5
    A = A_rand + bias
    A.requires_grad = True
    return A

def get_rho(A):
    rho_un = A @ A.mH
    trace = torch.trace(rho_un) + 1e-16
    return rho_un / trace

def partial_transpose(rho, dim):
    rho_reshaped = rho.view(dim, dim, dim, dim)
    rho_pt = rho_reshaped.permute(0, 3, 2, 1).contiguous()
    return rho_pt.view(dim * dim, dim * dim)

def get_metrics_high_dim(rho, dim):
    pt = partial_transpose(rho, dim)
    eigvals = torch.linalg.eigvalsh(pt)
    min_eig = torch.min(eigvals)
    negatives = eigvals[eigvals < 0]
    negativity_loss = torch.sum(torch.square(negatives)) 
    rho_reshaped = rho.view(dim, dim, dim, dim)
    rho_r = rho_reshaped.permute(0, 2, 1, 3).contiguous().view(dim*dim, dim*dim)
    r_norm = torch.linalg.matrix_norm(rho_r, ord='nuc')
    return min_eig, negativity_loss, r_norm

def try_repair_state(rho_np, dim):
    d2 = dim * dim
    I = np.eye(d2) / d2
    rho_tensor = rho_np.reshape(dim, dim, dim, dim)
    rho_pt = rho_tensor.transpose(0, 3, 2, 1).reshape(d2, d2)
    min_eig = np.min(np.linalg.eigvalsh(rho_pt))
    
    if min_eig >= 0: return rho_np
    
    p_needed = abs(min_eig) * d2 * 1.2 
    # 限制 p 的最大值，防止为了修复而把纠缠全毁了
    if p_needed > 0.1: return rho_np 
    
    rho_repaired = (1 - p_needed) * rho_np + p_needed * I
    return rho_repaired

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

# ================= 核心优化器 (封装版) =================

def solve_high_dim_optimized(dim, rank, max_attempts=30, base_seed=42):
    """
    针对特定 dim 和 rank 运行优化
    """
    print(f"\n🚀 [开始任务] Dim: {dim}x{dim}, Rank: {rank}")
    
    for attempt in range(1, max_attempts + 1):
        current_seed = base_seed + attempt * 100 # 种子间隔大一点
        set_seed(current_seed)
        
        A = get_ansatz_hybrid(dim, rank)
        optimizer = torch.optim.AdamW([A], lr=0.005, weight_decay=1e-5)
        
        # 针对不同维度动态调整步数
        total_steps = 20000 + (dim - 4) * 2000 
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=total_steps, eta_min=1e-5)
        
        best_candidate = None
        best_r_norm = 0
        best_min_eig = -1.0
        
        # 简化日志输出，只打印关键信息
        # print(f"   Attempt {attempt} (Seed {current_seed})...", end="\r")
        
        for i in range(total_steps):
            optimizer.zero_grad()
            rho = get_rho(A)
            min_eig, neg_loss, r_norm = get_metrics_high_dim(rho, dim)
            
            if min_eig.item() > -1e-5 and r_norm.item() > 1.0:
                if r_norm.item() > best_r_norm:
                    best_r_norm = r_norm.item()
                    best_min_eig = min_eig.item()
                    best_candidate = rho.detach().cpu().numpy()
            
            is_ppt_satisfied = min_eig.item() > -1e-6
            loss = 0.0
            
            if not is_ppt_satisfied:
                ppt_weight = 10000.0 * (1 + i/5000)
                loss += neg_loss * ppt_weight
                loss += -r_norm * 0.1
            else:
                loss += -r_norm * 50.0
                if min_eig < 0:
                    loss += neg_loss * 50000.0 
            
            loss.backward()
            optimizer.step()
            scheduler.step()
            
            if min_eig.item() > -1e-9 and r_norm.item() > 1.0001:
                print(f"   🎉 Attempt {attempt}: Found! PPT={min_eig.item():.2e}, CCN={r_norm.item():.5f}")
                return rho.detach().cpu().numpy()
        
        # 尝试修复
        if best_candidate is not None:
            rho_fixed = try_repair_state(best_candidate, dim)
            rho_t = torch.tensor(rho_fixed, device=device)
            m_eig, _, m_norm = get_metrics_high_dim(rho_t, dim)
            
            if m_eig.item() > -1e-15 and m_norm.item() > 1.0:
                print(f"   ✨ Attempt {attempt}: Repaired! PPT={m_eig.item():.2e}, CCN={m_norm.item():.5f}")
                return rho_fixed
    
    print(f"   ❌ Rank {rank} 尝试 {max_attempts} 次后未找到。")
    return None

# ================= 结果处理与保存函数 =================

def save_result(rho_np, dim, rank):
    """保存数据和图片"""
    base_name = f"BE_State_d{dim}_rank{rank}"
    
    # 1. 保存 NPY
    npy_path = os.path.join(OUTPUT_DIR, f"{base_name}.npy")
    np.save(npy_path, rho_np)
    
    # 2. 绘制并保存图片
    plt.figure(figsize=(12, 5))
    
    # 实部
    plt.subplot(1, 2, 1)
    sns.heatmap(rho_np.real, cmap="RdBu_r", center=0, square=True)
    
    # 计算 CCN 放在标题里
    rho_ts = rho_np.reshape(dim, dim, dim, dim)
    rho_r = rho_ts.transpose(0, 2, 1, 3).reshape(dim*dim, dim*dim)
    r_norm = np.sum(np.linalg.svd(rho_r, compute_uv=False))
    
    plt.title(f"Real Part (d={dim}, r={rank})\nCCN Norm={r_norm:.5f}")

    # 虚部
    plt.subplot(1, 2, 2)
    sns.heatmap(rho_np.imag, cmap="RdBu_r", center=0, square=True)
    plt.title("Imaginary Part")

    plt.tight_layout()
    img_path = os.path.join(OUTPUT_DIR, f"{base_name}.png")
    plt.savefig(img_path, dpi=150)
    plt.close() # 这一步很重要，批量跑防止内存溢出
    
    print(f"   💾 Saved: {npy_path}")
    print(f"   🖼️ Saved: {img_path}")

# ================= 主执行循环 =================

def main():
    print(f"📁 结果将保存在文件夹: ./{OUTPUT_DIR}/")
    print(f"📋 计划任务:")
    print(f"   Dimensions: {TARGET_DIMS}")
    print(f"   Ranks:      {TARGET_RANKS}")
    
    success_count = 0
    total_count = 0
    
    start_time_all = time.time()

    # 双重循环：遍历所有 Dimension 和 Rank 的组合
    for d in TARGET_DIMS:
        for r in TARGET_RANKS:
            total_count += 1
            
            # 运行优化器
            # max_attempts 可以根据需要调小一点以节省时间，或者调大以提高成功率
            rho = solve_high_dim_optimized(d, r, max_attempts=10)
            
            if rho is not None:
                save_result(rho, d, r)
                success_count += 1
            else:
                print(f"   ⚠️ 跳过 d={d}, r={r} (未找到)")
                
    total_time = time.time() - start_time_all
    print("\n" + "="*50)
    print(f"🏁 批量生成结束!")
    print(f"⏱️ 总耗时: {total_time:.2f} 秒")
    print(f"✅ 成功率: {success_count}/{total_count}")
    print(f"📁 所有文件已保存在: ./{OUTPUT_DIR}")
    print("="*50)

if __name__ == "__main__":
    main()

## 检测

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns

def check_bound_entanglement_in_notebook():
    # ================= 配置区 =================
    # 尝试从全局变量中获取数据，如果变量名不同，请手动修改这里
    try:
        # 优先使用 rho_final，如果它是None或不存在，尝试使用 rho_np
        if 'rho_final' in globals() and rho_final is not None:
            rho_input = rho_final
        elif 'rho_np' in globals() and rho_np is not None:
            rho_input = rho_np
        else:
            print("❌ 错误：未找到生成的密度矩阵变量 (rho_final 或 rho_np)。请先运行生成代码。")
            return

        # 获取维度
        if 'target_dim' in globals():
            d = target_dim
        else:
            # 尝试自动推断维度
            d = int(np.sqrt(rho_input.shape[0]))
            print(f"⚠️ 未找到 target_dim 变量，自动推断维度为 d={d}")
            
    except Exception as e:
        print(f"❌ 读取数据时出错: {e}")
        return
    # ==========================================

    print(f"🔬 正在分析 {d}x{d} 维量子态 (Matrix: {d**2}x{d**2})...\n")

    # 1. 格式标准化 (Tensor -> Numpy, 厄米化, 归一化)
    if isinstance(rho_input, torch.Tensor):
        rho = rho_input.detach().cpu().numpy()
    else:
        rho = np.array(rho_input, copy=True)
    
    # 强行厄米化 (消除计算误差)
    rho = (rho + rho.conj().T) / 2
    # 强行归一化
    rho = rho / np.trace(rho)

    d2 = d * d
    tolerance = 1e-9 # 浮点数容差

    # ---------------------------------------------------
    # 2. 判据 A: PPT (Positive Partial Transpose)
    # ---------------------------------------------------
    # 逻辑: 如果 min_eig < 0，则是NPT(自由纠缠)。如果 >= 0，则是PPT(可能是束缚纠缠)。
    rho_tensor = rho.reshape(d, d, d, d)
    # 转置子系统B (交换下标 1, 3)
    rho_pt = rho_tensor.transpose(0, 3, 2, 1).reshape(d2, d2)
    pt_evals = np.linalg.eigvalsh(rho_pt)
    min_pt_eig = np.min(pt_evals)

    is_ppt = min_pt_eig >= -tolerance

    # ---------------------------------------------------
    # 3. 判据 B: CCN (Computable Cross Norm / Realignment)
    # ---------------------------------------------------
    # 逻辑: 如果 CCN > 1，则一定是纠缠的。
    # 重排: indices (i, j, k, l) -> (i, k, j, l)
    rho_r = rho_tensor.transpose(0, 2, 1, 3).reshape(d2, d2)
    # 计算奇异值之和 (Trace Norm)
    singular_values = np.linalg.svd(rho_r, compute_uv=False)
    ccn_norm = np.sum(singular_values)

    is_entangled = ccn_norm > 1.0 + tolerance

    # ---------------------------------------------------
    # 4. 输出结果报告
    # ---------------------------------------------------
    print(f"1️⃣  [PPT 判据] 部分转置最小特征值: {min_pt_eig:.9f}")
    if is_ppt:
        print("    -> 结果: ✅ PPT (正部分转置)")
        print("    -> 含义: 该态不可被提纯。")
    else:
        print("    -> 结果: ❌ NPT (负部分转置)")
        print("    -> 含义: 该态是自由纠缠态 (Free Entangled)，不是束缚纠缠。")

    print(f"\n2️⃣  [CCN 判据] 重排范数 (Target > 1): {ccn_norm:.9f}")
    if is_entangled:
        print("    -> 结果: ✅ 确认纠缠")
    else:
        print("    -> 结果: ❓ 未检测到纠缠 (可能为可分态)")

    print("\n" + "="*40)
    if is_ppt and is_entangled:
        print("🏆 最终结论: 这是一个【束缚纠缠态】! 🏆")
    elif not is_ppt:
        print("结论: 这是一个【自由纠缠态】 (NPT Entangled)。")
    else:
        print("结论: 无法确认纠缠 (可能是PPT可分态)。")
    print("="*40)

    # ---------------------------------------------------
    # 5. 可视化绘图
    # ---------------------------------------------------
    plt.figure(figsize=(12, 4))
    
    # 图1: 部分转置谱 (PT Eigenvalues)
    plt.subplot(1, 2, 1)
    plt.plot(pt_evals, 'b.-', markersize=3, label='Eigenvalues')
    plt.axhline(0, color='r', linestyle='--', linewidth=1, label='Zero')
    plt.title(f"PT Spectrum (Min: {min_pt_eig:.1e})")
    plt.legend()
    plt.grid(True, alpha=0.3)
    if min_pt_eig < 0:
        # 如果有负值，放大显示负值区域
        plt.ylim(min_pt_eig * 1.5, abs(min_pt_eig) * 1.5)
        plt.title(f"PT Spectrum (Zoomed on Negative)")

    # 图2: 密度矩阵实部
    plt.subplot(1, 2, 2)
    # 为了显示清晰，取对数或者限制范围
    sns.heatmap(rho.real, cmap="RdBu_r", center=0, vmin=-0.01, vmax=0.01)
    plt.title("Density Matrix (Real Part)")
    
    plt.tight_layout()
    plt.show()

# 运行检验
check_bound_entanglement_in_notebook()

In [ ]:
import numpy as np

def check_bound_entanglement(rho, dimA=None, dimB=None, tol=1e-8):
    """
    判断一个密度矩阵是否为两体束缚纠缠态 (Bipartite Bound Entangled State)
    """
    print("="*50)
    print("🔬 量子态束缚纠缠检测程序启动...")
    print("="*50)

    # 1. 维度推断
    dim = rho.shape[0]
    if dimA is None or dimB is None:
        dimA = int(np.sqrt(dim))
        dimB = dim // dimA
        if dimA * dimB != dim:
            raise ValueError("矩阵维度无法自动分解为两个子系统，请手动指定 dimA 和 dimB。")
    print(f"✅ 系统维度: {dim}x{dim} (推断为子系统A: {dimA}, 子系统B: {dimB})")

    # 2. 合法性检查 (迹为1，半正定)
    trace = np.trace(rho)
    if abs(trace - 1.0) > tol:
        print(f"⚠️ 警告: 矩阵的迹为 {trace.real:.6f}，不严格等于 1，但程序将继续计算。")
    
    eigenvalues_rho = np.linalg.eigvalsh(rho)
    if np.min(eigenvalues_rho) < -tol:
        print(f"❌ 错误: 矩阵存在负特征值 {np.min(eigenvalues_rho):.6e}，不是合法的量子态！")
        return

    # 3. 计算偏转置 (Partial Transpose)
    # 将矩阵 reshape 为4阶张量 (A_row, B_row, A_col, B_col)
    rho_tensor = rho.reshape((dimA, dimB, dimA, dimB))
    # 对 B 系统进行偏转置 (交换 B_row 和 B_col，即索引 1 和 3)
    rho_pt_tensor = rho_tensor.transpose((0, 3, 2, 1))
    rho_pt = rho_pt_tensor.reshape((dim, dim))

    # 计算偏转置后的特征值
    eigenvalues_pt = np.linalg.eigvalsh(rho_pt)
    min_eig_pt = np.min(eigenvalues_pt)
    print(f"📊 偏转置矩阵最小特征值: {min_eig_pt:.8e}")

    # 4. 判断 PPT (正偏转置) 还是 NPT (负偏转置)
    is_ppt = min_eig_pt >= -tol

    if not is_ppt:
        print("\n🚫 【最终结论】：它不是束缚纠缠态！")
        print("原因：该态具有负偏转置 (NPT)。这说明它是可以被直接提纯的【自由纠缠态】(Free Entangled State)。")
        return

    print("✅ 该态具有正偏转置 (PPT)。它满足束缚纠缠的第一个条件。")
    print("⏳ 正在计算 CCNR 判据 (交叉范数重排) 以检测是否存在纠缠...")

    # 5. 计算 CCNR 判据 (检测 PPT 态是否真正纠缠)
    # 重排操作 (Realignment): 交换 A_col 和 B_row (即索引 1 和 2)
    R_tensor = rho_tensor.transpose((0, 2, 1, 3))
    R_matrix = R_tensor.reshape((dimA * dimA, dimB * dimB))
    
    # 计算奇异值并求和 (迹范数)
    singular_values = np.linalg.svd(R_matrix, compute_uv=False)
    ccnr_value = np.sum(singular_values)
    print(f"📊 CCNR 判据值 (Trace Norm): {ccnr_value:.6f} (阈值为 1.0)")

    # 6. 综合得出最终结论
    if ccnr_value > 1.0 + tol:
        print("\n🎉 【最终结论】：绝对确认！这是一个【两体束缚纠缠态】(Bound Entangled State)！")
        print("原因：它同时满足 PPT（无法提纯）且 CCNR > 1（证明存在纠缠）。")
    else:
        print("\n⚠️ 【最终结论】：目前证据不足以证明它是束缚纠缠态。")
        print("原因：它是 PPT 态，但 CCNR 判据值 <= 1。这意味着它大概率已经退化成了无纠缠的【可分态】(Separable State)。")
        print("注：如果它是你在论文中看到的特殊构造态，CCNR判据可能失效，你需要借助针对该态专门设计的纠缠目击者(Entanglement Witness)矩阵来做最终判定。")


# ==========================================
# 在这里输入你的量子态矩阵 rho
# ==========================================
if __name__ == "__main__":
    # 请把你跑出的那段 np.array([...]) 直接粘贴在下面：
    rho = np.array(
    [[ 0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ],
     [ 0.j           ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ],
     [ 0.j           ,  0.j           ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.j           ],
     [ 0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ],
     [ 0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ],
     [ 0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ],
     [ 0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.j           ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ],
     [ 0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.j           ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ],
     [ 0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.j           ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ],
     [ 0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.j           ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ],
     [ 0.j           ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ],
     [ 0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ],
     [ 0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ],
     [ 0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.0833+0.j      ,  0.j           ,  0.0833+0.j      ],
     [ 0.j           ,  0.j           ,  0.0833+0.j      ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.j           ],
     [ 0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.j           ,  0.0833+0.j      ,  0.j           ,  0.0833+0.j      ]]
)

    # 调用检测程序
    check_bound_entanglement(rho)

🔬 量子态束缚纠缠检测程序启动...
✅ 系统维度: 16x16 (推断为子系统A: 4, 子系统B: 4)
⚠️ 警告: 矩阵的迹为 1.332800，不严格等于 1，但程序将继续计算。
❌ 错误: 矩阵存在负特征值 -5.148223e-02，不是合法的量子态！


In [ ]:
import torch
import numpy as np
import os
import re

# ================= 0. 配置与环境 =================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_dtype(torch.float64) 
print(f"🚀 计算设备: {device}")

# ================= 1. 自动扫描文件 =================
def scan_and_load_files():
    """
    自动扫描当前目录下所有的 .npy 文件，尝试推断其维度 d。
    支持的文件名格式例如: BE_State_d4_rank4.npy, BE_State_4x4_Rank4.npy 等
    """
    file_list = []
    for f in os.listdir('.'):
        if f.endswith('.npy'):
            # 尝试用正则提取维度，比如找 "d4" 或者 "4x4"
            match = re.search(r'd(\d+)|(\d+)x\d+', f, re.IGNORECASE)
            if match:
                d = int(match.group(1) or match.group(2))
                file_list.append((f, d))
    
    if not file_list:
        print("❌ 警告：当前目录下未找到任何包含维度信息的 .npy 文件！")
        print("   请确保文件与代码在同一文件夹，或者手动修改文件名。")
    return sorted(file_list, key=lambda x: x[1])

# ================= 2. 物理基础函数 =================
def load_state_smart(filename, dim):
    try:
        rho = np.load(filename)
        rho = torch.tensor(rho, device=device, dtype=torch.complex128)
        # 验证大小是否匹配 dim*dim x dim*dim
        if rho.shape[0] != dim * dim: return None
        rho = (rho + rho.mH) / 2.0 # 强行厄米化
        rho = rho / torch.trace(rho) # 归一化
        return rho
    except Exception as e:
        print(f"   [加载出错] {e}")
        return None

def partial_transpose_torch(rho, dim):
    rho_ts = rho.view(dim, dim, dim, dim)
    return rho_ts.permute(0, 3, 2, 1).contiguous().view(dim*dim, dim*dim)

# ================= 3. 五大判据实现 =================
def criterion_realignment(rho, dim):
    rho_ts = rho.view(dim, dim, dim, dim)
    rho_r = rho_ts.permute(0, 2, 1, 3).contiguous().view(dim*dim, dim*dim)
    ccn_norm = torch.linalg.svdvals(rho_r).sum().real.item()
    return ccn_norm, ccn_norm > 1.0 + 1e-8

def criterion_quasi_pure(rho, dim):
    evals, evecs = torch.linalg.eigh(rho)
    lambda_0 = evals[-1].real
    dominant_vec = evecs[:, -1].view(dim, dim)
    sum_other_lambdas = torch.sum(evals[:-1]).real
    S_i = torch.linalg.svdvals(dominant_vec)
    C_pure = torch.sqrt(2.0 * (1.0 - torch.sum(S_i**4))).real
    C_qp = (lambda_0 * C_pure - sum_other_lambdas).item()
    return C_qp, C_qp > 1e-8

def criterion_correlation_tensor(rho, dim):
    rho_ts = rho.view(dim, dim, dim, dim)
    rho_A = torch.einsum('ijik->jk', rho_ts)
    rho_B = torch.einsum('ijil->jl', rho_ts)
    purity = torch.trace(rho @ rho).real
    purity_A = torch.trace(rho_A @ rho_A).real
    purity_B = torch.trace(rho_B @ rho_B).real
    corr_val = (purity - purity_A - purity_B + 1.0).item()
    return corr_val, corr_val > 1e-6

def criterion_range(rho, dim):
    rho_pt = partial_transpose_torch(rho, dim)
    rank_rho = torch.linalg.matrix_rank(rho, rtol=1e-8).item()
    rank_pt = torch.linalg.matrix_rank(rho_pt, rtol=1e-8).item()
    is_suspicious = (rank_rho < dim*dim) and (rank_pt > rank_rho)
    return rank_rho, rank_pt, is_suspicious

class WitnessOptimizer:
    def __init__(self, dim):
        self.params = [torch.randn(dim, dim, device=device, requires_grad=True) for _ in range(4)]
        self.opt = torch.optim.Adam(self.params, lr=0.05)
        D = dim * dim
        I = torch.eye(D, dtype=torch.complex128, device=device)
        V = torch.zeros((D, D), dtype=torch.complex128, device=device)
        for i in range(dim):
            for j in range(dim): V[i*dim+j, j*dim+i] = 1.0
        self.W_ind = I/dim - (I-V)/2.0

    def get_max_violation(self, rho, steps=100):
        max_v = -float('inf')
        for _ in range(steps):
            self.opt.zero_grad()
            UA = torch.matrix_exp(torch.complex(self.params[0]-self.params[0].T, self.params[1]+self.params[1].T))
            UB = torch.matrix_exp(torch.complex(self.params[2]-self.params[2].T, self.params[3]+self.params[3].T))
            rho_rot = torch.kron(UA, UB) @ rho @ torch.kron(UA, UB).mH
            v = -torch.real(torch.trace(self.W_ind @ rho_rot))
            (-v).backward(); self.opt.step()
            if v.item() > max_v: max_v = v.item()
        return max_v, max_v > 1e-6

def combined_fast_scanner(rho, dim):
    ccn_val, is_ccn = criterion_realignment(rho, dim)
    if is_ccn: return f"✅ CCN 成功检出 (CCN={ccn_val:.4f} > 1)"
    qp_val, is_qp = criterion_quasi_pure(rho, dim)
    if is_qp: return f"✅ 准纯近似捡漏 (C_qp={qp_val:.4e} > 0)"
    return f"❌ 联合扫描失败 (CCN={ccn_val:.4f}, C_qp={qp_val:.4e})"

# ================= 4. 主执行循环 =================

if __name__ == "__main__":
    print(f"\n{'='*60}")
    print("🔬 启动量子纠缠五大判据全面检验体系")
    print("="*60)

    files_to_test = scan_and_load_files()
    
    if files_to_test:
        print(f"📂 发现 {len(files_to_test)} 个合法样本文件。开始逐一检验...\n")

    for filename, d in files_to_test:
        rho = load_state_smart(filename, d)
        if rho is None: 
            print(f"⚠️ 文件 {filename} 维度不匹配或已损坏，跳过。")
            continue
            
        print(f"\n📁 正在分析文件: {filename} (Dim={d}x{d})")
        print("-" * 60)
        
        ccn_val, e1 = criterion_realignment(rho, d)
        print(f" 1. [重排判据/CCN]     值: {ccn_val:.6f} {'✅ 纠缠' if e1 else '❌ 失败'}")
        
        qp_val, e2 = criterion_quasi_pure(rho, d)
        print(f" 2. [准纯近似/C_qp]    值: {qp_val:.2e} {'✅ 纠缠' if e2 else '❌ 失败'}")
        
        corr_val, e3 = criterion_correlation_tensor(rho, d)
        print(f" 3. [关联张量代数]     值: {corr_val:.6f} {'✅ 疑似' if e3 else '❌ 失败'}")
        
        rr, rpt, e4 = criterion_range(rho, d)
        print(f" 4. [值域判据/Rank]    Rank(ρ)={rr:2d}, Rank(PT)={rpt:2d} {'💡 疑似BE态' if e4 else '➖ 正常'}")
        
        nd_val, e5 = WitnessOptimizer(d).get_max_violation(rho, steps=120)
        print(f" 5. [不可分解/Nd-EW]   最大违反 V={nd_val:.6f} {'✅ 纠缠' if e5 else '❌ 失败'}")
        
        print(f" 🌟 [联合快速扫描]     {combined_fast_scanner(rho, d)}")
        print("="*60)

🚀 计算设备: cpu

🔬 启动量子纠缠五大判据全面检验体系
📂 发现 115 个合法样本文件。开始逐一检验...


📁 正在分析文件: BE_State_4x4_Rank10_0.npy (Dim=4x4)
------------------------------------------------------------
 1. [重排判据/CCN]     值: 1.218682 ✅ 纠缠
 2. [准纯近似/C_qp]    值: -6.05e-01 ❌ 失败
 3. [关联张量代数]     值: 0.626154 ✅ 疑似
 4. [值域判据/Rank]    Rank(ρ)=10, Rank(PT)=16 💡 疑似BE态
 5. [不可分解/Nd-EW]   最大违反 V=0.248256 ✅ 纠缠
 🌟 [联合快速扫描]     ✅ CCN 成功检出 (CCN=1.2187 > 1)

📁 正在分析文件: BE_State_4x4_Rank10_1.npy (Dim=4x4)
------------------------------------------------------------
 1. [重排判据/CCN]     值: 1.011963 ✅ 纠缠
 2. [准纯近似/C_qp]    值: -6.15e-01 ❌ 失败
 3. [关联张量代数]     值: 0.544073 ✅ 疑似
 4. [值域判据/Rank]    Rank(ρ)=10, Rank(PT)=16 💡 疑似BE态
 5. [不可分解/Nd-EW]   最大违反 V=0.238808 ✅ 纠缠
 🌟 [联合快速扫描]     ✅ CCN 成功检出 (CCN=1.0120 > 1)

📁 正在分析文件: BE_State_4x4_Rank10_2.npy (Dim=4x4)
------------------------------------------------------------
 1. [重排判据/CCN]     值: 1.020727 ✅ 纠缠
 2. [准纯近似/C_qp]    值: -6.38e-01 ❌ 失败
 3. [关联张量代数]     值: 0.566828 ✅ 疑似
 4. [值域判据/Rank]    Rank(ρ

In [6]:
import numpy as np
import torch

# ==========================================
# 内部依赖：PyTorch 驱动的不可分解目击者优化器
# ==========================================
class NdEWOptimizer:
    def __init__(self, dim, tol=1e-8):
        self.dim = dim
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.tol = tol
        
        # 构造不可分解目击者 W = I/d - P_asym
        D = dim * dim
        I = torch.eye(D, dtype=torch.complex128, device=self.device)
        V = torch.zeros((D, D), dtype=torch.complex128, device=self.device)
        for i in range(dim):
            for j in range(dim):
                V[i*dim + j, j*dim + i] = 1.0
        self.W_ind = I/dim - (I - V)/2.0
        
        # 初始化幺正变换参数
        self.params = [torch.randn(dim, dim, device=self.device, requires_grad=True) for _ in range(4)]
        self.opt = torch.optim.Adam(self.params, lr=0.05)

    def get_max_violation(self, rho_np, steps=150):
        rho = torch.tensor(rho_np, device=self.device, dtype=torch.complex128)
        max_v = -float('inf')
        for _ in range(steps):
            self.opt.zero_grad()
            UA = torch.matrix_exp(torch.complex(self.params[0]-self.params[0].T, self.params[1]+self.params[1].T))
            UB = torch.matrix_exp(torch.complex(self.params[2]-self.params[2].T, self.params[3]+self.params[3].T))
            rho_rot = torch.kron(UA, UB) @ rho @ torch.kron(UA, UB).mH
            v = -torch.real(torch.trace(self.W_ind @ rho_rot))
            (-v).backward()
            self.opt.step()
            if v.item() > max_v: max_v = v.item()
        return max_v

# ==========================================
# 核心主程序：量子态五大判据综合检验
# ==========================================
def check_bound_entanglement(rho, dimA=None, dimB=None, tol=1e-8):
    """
    判断一个密度矩阵是否为两体束缚纠缠态 (Bipartite Bound Entangled State)
    """
    print("="*60)
    print("🔬 量子态束缚纠缠全面检测程序启动...")
    print("="*60)

    # ---------------------------------------------------
    # 1. 维度推断与合法性检查
    # ---------------------------------------------------
    dim = rho.shape[0]
    if dimA is None or dimB is None:
        dimA = int(np.sqrt(dim))
        dimB = dim // dimA
        if dimA * dimB != dim:
            raise ValueError("矩阵维度无法自动分解，请手动指定 dimA 和 dimB。")
    print(f"✅ 系统维度: {dim}x{dim} (子系统A: {dimA}, 子系统B: {dimB})")

    rho = (rho + rho.conj().T) / 2.0 # 强行厄米化，消除计算误差
    trace = np.trace(rho).real
    if abs(trace - 1.0) > tol:
        print(f"⚠️ 警告: 矩阵的迹为 {trace:.6f}，已自动归一化。")
        rho = rho / trace
    
    eigenvalues_rho = np.linalg.eigvalsh(rho)
    if np.min(eigenvalues_rho) < -tol:
        print(f"❌ 错误: 矩阵存在负特征值 {np.min(eigenvalues_rho):.6e}，不是合法的量子态！")
        return

    # 张量化备用
    rho_tensor = rho.reshape((dimA, dimB, dimA, dimB))

    # ---------------------------------------------------
    # 2. 前置判定：PPT 判据
    # ---------------------------------------------------
    rho_pt_tensor = rho_tensor.transpose((0, 3, 2, 1))
    rho_pt = rho_pt_tensor.reshape((dim, dim))
    min_eig_pt = np.min(np.linalg.eigvalsh(rho_pt))
    is_ppt = min_eig_pt >= -tol
    
    print("\n" + "-"*40)
    print(f"0️⃣ [前置属性] PPT 判据 (部分转置最小特征值: {min_eig_pt:.2e})")
    if not is_ppt:
        print("   -> 🚫 结论：负部分转置 (NPT)。")
        print("   -> 物理意义：该态为【自由纠缠态】(Free Entangled)，绝对不是束缚纠缠！")
        print("   -> (后续判据将继续执行，以展示其纠缠强度的不同侧面)")
    else:
        print("   -> ✅ 结论：正部分转置 (PPT)。满足束缚纠缠的第一必要条件！")

    # ---------------------------------------------------
    # 3. 方法一：重排判据 (CCN / Realignment)
    # ---------------------------------------------------
    R_tensor = rho_tensor.transpose((0, 2, 1, 3))
    R_matrix = R_tensor.reshape((dimA * dimA, dimB * dimB))
    ccn_value = np.sum(np.linalg.svd(R_matrix, compute_uv=False))
    pass_ccn = ccn_value > 1.0 + tol
    
    print(f"\n1️⃣ [代数判据] 重排判据 CCN Norm: {ccn_value:.6f} (阈值>1.0)")
    print(f"   -> {'✅ 发现纠缠！' if pass_ccn else '❌ 未发现'}")

    # ---------------------------------------------------
    # 4. 方法二：准纯近似 (Quasi-Pure Approximation)
    # ---------------------------------------------------
    evals, evecs = np.linalg.eigh(rho)
    dom_vec = evecs[:, -1].reshape((dimA, dimB))
    s_vals = np.linalg.svd(dom_vec, compute_uv=False)
    c_pure = np.sqrt(2 * (1.0 - np.sum(s_vals**4)))
    c_qp = evals[-1] * c_pure - np.sum(evals[:-1])
    pass_qp = c_qp > tol
    
    print(f"\n2️⃣ [谱系判据] 准纯近似 C_qp: {c_qp:.4e} (阈值>0.0)")
    print(f"   -> {'✅ 发现纠缠！' if pass_qp else '❌ 未发现 (该态混合度可能过高)'}")

    # ---------------------------------------------------
    # 5. 特别结合：重排与准纯近似联合雷达
    # ---------------------------------------------------
    print(f"\n🌟 [联合扫描] 重排判据 与 准纯近似 联合探测结果：")
    if pass_ccn:
        print("   -> ✅ CCN 强力检出！(无需准纯近似补充)")
    elif pass_qp:
        print("   -> ✅ 准纯近似成功“捡漏”！(弥补了重排判据对特定弱混合态的失效)")
    else:
        print("   -> ❌ 两大主流代数判据均告失效，纠缠隐蔽性极深。")

    # ---------------------------------------------------
    # 6. 方法三：关联张量判据 (Correlation Tensor)
    # ---------------------------------------------------
    rho_A = np.trace(rho_tensor, axis1=1, axis2=3)
    rho_B = np.trace(rho_tensor, axis1=0, axis2=2)
    corr_val = np.trace(rho@rho).real - np.trace(rho_A@rho_A).real - np.trace(rho_B@rho_B).real + 1.0
    pass_corr = corr_val > 1e-6
    
    print(f"\n3️⃣ [偏迹判据] 关联张量代数和: {corr_val:.6f}")
    print(f"   -> {'✅ 发现异常关联！' if pass_corr else '❌ 未发现'}")

    # ---------------------------------------------------
    # 7. 方法四：值域判据 (Range Criterion)
    # ---------------------------------------------------
    rank_rho = np.linalg.matrix_rank(rho, tol=1e-8)
    rank_pt = np.linalg.matrix_rank(rho_pt, tol=1e-8)
    pass_range = (rank_rho < dim) and (rank_pt > rank_rho)
    
    print(f"\n4️⃣ [几何判据] 值域判据 Rank(ρ)={rank_rho}, Rank(ρ_PT)={rank_pt}")
    if pass_range:
        print("   -> 💡 发现值域断裂！极大概率为束缚纠缠态。")
    else:
        print("   -> ➖ 无明显秩异常 (该判据常用于解析证明，数值探测较弱)。")

    # ---------------------------------------------------
    # 8. 方法五：不可分解纠缠目击者 (Nd-EW)
    # ---------------------------------------------------
    pass_ew = False
    if dimA == dimB:
        print(f"\n5️⃣ [物理观测] 正在启动不可分解目击者 (Nd-EW) 梯度优化探测...")
        opt = NdEWOptimizer(dimA)
        max_v = opt.get_max_violation(rho, steps=150)
        pass_ew = max_v > 1e-6
        print(f"   -> 最大违背度 V: {max_v:.6f} (阈值>0.0)")
        print(f"   -> {'✅ 强力探测到纠缠信号！' if pass_ew else '❌ 未探测到'}")
    else:
        print(f"\n5️⃣ [物理观测] Nd-EW 仅支持方阵系统 (dimA=dimB)，跳过。")

    # ---------------------------------------------------
    # 9. 综合判定结论
    # ---------------------------------------------------
    print("\n" + "="*60)
    
    # 只要任何一个纠缠判据生效，即认为探测到纠缠
    is_entangled = pass_ccn or pass_qp or pass_corr or pass_ew
    
    if not is_ppt:
        print("💡 终极结论：这是一个【自由纠缠态】(Free Entangled State)。")
        print("   说明：态中存在可通过局部操作提纯的自由纠缠资源。")
    elif is_ppt and is_entangled:
        print("🏆 终极结论：绝对确认！这是一个【两体束缚纠缠态】(Bound Entangled State)！")
        print("   说明：它满足 PPT 条件（无法提纯），但被高级判据成功抓取到了内部的隐蔽纠缠。")
    else:
        print("⚠️ 终极结论：目前证据不足以证明它是纠缠态。")
        print("   说明：它是 PPT 态，且逃过了目前所有五大判据的检测。它大概率是【可分态】(Separable State)。")
    print("="*60)


# ==========================================
# 在这里输入你的量子态矩阵 rho
# ==========================================
if __name__ == "__main__":
    # 你跑出的矩阵
    rho = np.array(
[
 [ 0.005643+0.j      , -0.004569-0.004139j, -0.005896+0.001096j, -0.006448+0.000284j, -0.000391-0.008853j, -0.007622+0.006081j,  0.002677+0.009991j, -0.002221+0.014269j, -0.003899-0.003902j,  0.003910+0.004436j,  0.004359+0.002026j,  0.004440-0.001624j, -0.004572+0.001811j,  0.003507+0.000189j,  0.004161-0.002601j,  0.002844-0.001628j],
 [-0.004569+0.004139j,  0.009597+0.j      ,  0.003630-0.006227j,  0.006479-0.009447j,  0.007517+0.009220j,  0.001271-0.016528j, -0.009713+0.000043j, -0.019305-0.016265j,  0.003473-0.001646j, -0.006261-0.000762j, -0.010301+0.000165j, -0.004178+0.006519j,  0.002036-0.003131j, -0.004703+0.002522j, -0.001264+0.008482j, -0.000917+0.006463j],
 [-0.005896-0.001096j,  0.003630+0.006227j,  0.026319+0.j      ,  0.007491+0.003818j,  0.008214+0.006147j,  0.013318-0.005078j,  0.014952-0.064646j,  0.007431-0.026834j,  0.000127+0.019237j, -0.002174-0.006342j, -0.019797+0.014122j,  0.009284+0.002989j,  0.012455-0.001996j, -0.003179-0.002319j, -0.006975+0.005662j, -0.004356-0.006448j],
 [-0.006448-0.000284j,  0.006479+0.009447j,  0.007491-0.003818j,  0.017458+0.j      , -0.004800+0.013984j,  0.019780-0.011511j, -0.015421-0.013214j,  0.002844-0.036497j,  0.007484-0.003553j, -0.003832-0.005480j,  0.000012-0.001986j, -0.013084-0.004310j,  0.001393-0.001516j, -0.005511-0.003622j, -0.005774+0.001541j, -0.009980+0.005854j],
 [-0.000391+0.008853j,  0.007517-0.009220j,  0.008214-0.006147j, -0.004800-0.013984j,  0.027816+0.j      , -0.015738-0.014685j, -0.010777-0.026958j, -0.026540-0.002028j, -0.006493+0.007562j, -0.007495+0.003441j,  0.008930+0.003478j,  0.005608+0.023271j,  0.006486-0.004694j, -0.002053+0.007908j, -0.006958-0.001183j,  0.012002+0.002677j],
 [-0.007622-0.006081j,  0.001271+0.016528j,  0.013318+0.005078j,  0.019780+0.011511j, -0.015738+0.014685j,  0.032894+0.j      ,  0.004010-0.019362j,  0.028150-0.042483j,  0.007487+0.003176j,  0.000061-0.009664j, -0.012131-0.012533j, -0.009407-0.010254j,  0.004002+0.001262j, -0.003891-0.008509j, -0.011505+0.003418j, -0.014686-0.001758j],
 [ 0.002677-0.009991j, -0.009713-0.000043j,  0.014952+0.064646j, -0.015421+0.013214j, -0.010777+0.026958j,  0.004010+0.019362j,  0.296717+0.j      ,  0.033665+0.035050j, -0.042889-0.018181j,  0.019411+0.006614j, -0.168458+0.094733j,  0.018253-0.011798j, -0.005487+0.031691j,  0.009329-0.009133j,  0.070882+0.021487j, -0.000452-0.009446j],
 [-0.002221-0.014269j, -0.019305+0.016265j,  0.007431+0.026834j,  0.002844+0.036497j, -0.026540+0.002028j,  0.028150+0.042483j,  0.033665-0.035050j,  0.092326+0.j      , -0.004217+0.016061j,  0.010490-0.012680j,  0.034446-0.019949j, -0.006963-0.012993j,  0.008004+0.008328j,  0.005275-0.011123j, -0.018005-0.031093j, -0.005158-0.016955j],
 [-0.003899+0.003902j,  0.003473+0.001646j,  0.000127-0.019237j,  0.007484+0.003553j, -0.006493-0.007562j,  0.007487-0.003176j, -0.042889+0.018181j, -0.004217-0.016061j,  0.036474+0.j      , -0.009373+0.002161j, -0.032974-0.005941j,  0.014405-0.017181j, -0.004184-0.019072j,  0.000986+0.001281j,  0.002739+0.030217j, -0.014500-0.005745j],
 [ 0.003910-0.004436j, -0.006261+0.000762j, -0.002174+0.006342j, -0.003832+0.005480j, -0.007495-0.003441j,  0.000061+0.009664j,  0.019411-0.006614j,  0.010490+0.012680j, -0.009373-0.002161j,  0.008246+0.j      ,  0.007900+0.014867j, -0.001784-0.007868j, -0.002359+0.007723j,  0.002276-0.003637j,  0.007903-0.010589j, -0.000371-0.001206j],
 [ 0.004359-0.002026j, -0.010301-0.000165j, -0.019797-0.014122j,  0.000012+0.001986j,  0.008930-0.003478j, -0.012131+0.012533j, -0.168458-0.094733j,  0.034446+0.019949j, -0.032974+0.005941j,  0.007900-0.014867j,  0.290430+0.j      , -0.072396+0.016857j,  0.010178+0.013952j, -0.007704+0.000316j, -0.041219-0.132326j,  0.022636+0.024376j],
 [ 0.004440+0.001624j, -0.004178-0.006519j,  0.009284-0.002989j, -0.013084+0.004310j,  0.005608-0.023271j, -0.009407+0.010254j,  0.018253+0.011798j, -0.006963+0.012993j,  0.014405+0.017181j, -0.001784+0.007868j, -0.072396-0.016857j,  0.046028+0.j      ,  0.001120-0.011854j,  0.008201+0.003349j, -0.001243+0.041486j,  0.000069-0.021378j],
 [-0.004572-0.001811j,  0.002036+0.003131j,  0.012455+0.001996j,  0.001393+0.001516j,  0.006486+0.004694j,  0.004002-0.001262j, -0.005487-0.031691j,  0.008004-0.008328j, -0.004184+0.019072j, -0.002359-0.007723j,  0.010178-0.013952j,  0.001120+0.011854j,  0.013116+0.j      , -0.003486+0.000642j, -0.017220-0.004415j,  0.004217-0.003435j],
 [ 0.003507-0.000189j, -0.004703-0.002522j, -0.003179+0.002319j, -0.005511+0.003622j, -0.002053-0.007908j, -0.003891+0.008509j,  0.009329+0.009133j,  0.005275+0.011123j,  0.000986-0.001281j,  0.002276+0.003637j, -0.007704-0.000316j,  0.008201-0.003349j, -0.003486-0.000642j,  0.004268+0.j      ,  0.003702+0.002903j,  0.000578-0.004918j],
 [ 0.004161+0.002601j, -0.001264-0.008482j, -0.006975-0.005662j, -0.005774-0.001541j, -0.006958+0.001183j, -0.011505-0.003418j,  0.070882-0.021487j, -0.018005+0.031093j,  0.002739-0.030217j,  0.007903+0.010589j, -0.041219+0.132326j, -0.001243-0.041486j, -0.017220+0.004415j,  0.003702-0.002903j,  0.077344+0.j      , -0.013667+0.010378j],
 [ 0.002844+0.001628j, -0.000917-0.006463j, -0.004356+0.006448j, -0.009980-0.005854j,  0.012002-0.002677j, -0.014686+0.001758j, -0.000452+0.009446j, -0.005158+0.016955j, -0.014500+0.005745j, -0.000371+0.001206j,  0.022636-0.024376j,  0.000069+0.021378j,  0.004217+0.003435j,  0.000578+0.004918j, -0.013667-0.010378j,  0.015323+0.j      ]
 ] )  # 调用检测程序
    check_bound_entanglement(rho)

🔬 量子态束缚纠缠全面检测程序启动...
✅ 系统维度: 16x16 (子系统A: 4, 子系统B: 4)
⚠️ 警告: 矩阵的迹为 0.999999，已自动归一化。

----------------------------------------
0️⃣ [前置属性] PPT 判据 (部分转置最小特征值: -9.89e-03)
   -> 🚫 结论：负部分转置 (NPT)。
   -> 物理意义：该态为【自由纠缠态】(Free Entangled)，绝对不是束缚纠缠！
   -> (后续判据将继续执行，以展示其纠缠强度的不同侧面)

1️⃣ [代数判据] 重排判据 CCN Norm: 1.037375 (阈值>1.0)
   -> ✅ 发现纠缠！

2️⃣ [谱系判据] 准纯近似 C_qp: -3.1584e-01 (阈值>0.0)
   -> ❌ 未发现 (该态混合度可能过高)

🌟 [联合扫描] 重排判据 与 准纯近似 联合探测结果：
   -> ✅ CCN 强力检出！(无需准纯近似补充)

3️⃣ [偏迹判据] 关联张量代数和: 0.295169
   -> ✅ 发现异常关联！

4️⃣ [几何判据] 值域判据 Rank(ρ)=16, Rank(ρ_PT)=16
   -> ➖ 无明显秩异常 (该判据常用于解析证明，数值探测较弱)。

5️⃣ [物理观测] 正在启动不可分解目击者 (Nd-EW) 梯度优化探测...


RuntimeError: expected m1 and m2 to have the same dtype, but got: struct c10::complex<float> != struct c10::complex<double>